In [ ]:
import os
from datetime import datetime, timedelta
import pandas as pd

## Diccionarios que simularán nuestras bases de datos temporales.
usuarios_db = {}
items_db = {}
prestamos_db = []
ventas_db = []

# Credenciales preconfiguradas para el módulo de Administración
ADMIN_USER = "admin"
ADMIN_PASS = "1234"

## funciones de validacion

def validar_nombre_apellido(texto):
    """Valida que el texto no tenga menos de 3 letras y no contenga números."""
    if len(texto) < 3:
        return False
    if any(char.isdigit() for char in texto):
        return False
    return True


def validar_documento(doc):
    """Valida que el documento tenga entre 3 y 15 dígitos y solo contenga números."""
    if not doc.isdigit():
        return False
    if not (3 <= len(doc) <= 15):
        return False
    return True


def validar_correo(correo):
    """Valida de forma básica que contenga '@' y termine en '.com'."""
    if "@" in correo and correo.endswith(".com"):
        return True
    return False


def limpiar_pantalla():
    """Limpia la consola según el sistema operativo."""
    os.system("cls" if os.name == "nt" else "clear")

## registro de usuario

def registrar_usuario():
    """Solicita los datos del nuevo amigo, aplica las validaciones requeridas
    y lo almacena los datos"""
    limpiar_pantalla()
    print("--- REGISTRAR NUEVO AMIGO ---")

    nombre = input("Nombre: ").strip()
    if not validar_nombre_apellido(nombre):
        print("El nombre debe tener mínimo 3 letras y no contener números")
        return

    apellido = input("Apellido: ").strip()
    if not validar_nombre_apellido(apellido):
        print("El apellido debe tener mínimo 3 letras y no contener números")
        return

    documento = input("Documento de identidad: ").strip()
    if not validar_documento(documento):
        print("El documento debe ser numérico y tener entre 3 y 15 dígitos")
        return

    if documento in usuarios_db:
        print("Ya existe un usuario registrado con este documento")
        return

    correo = input("Correo electrónico: ").strip()
    if not validar_correo(correo):
        print("Error: Correo inválido. Debe contener '@' y terminar en '.com'.")
        return

    print("Opciones de tiempo de préstamo disponibles: 5, 10, 15 o 30 días.")
    try:
        dias = int(input("Seleccione los días permitidos: "))
        if dias not in [5, 10, 15, 30]:
            print("Opción de días no válida")
            return
    except ValueError:
        print("Debe ingresar un número entero")
        return

    # Guardamos la información
    usuarios_db[documento] = pd.Series(
        {
            "Nombre": nombre,
            "Apellido": apellido,
            "Correo": correo,
            "Dias_Permitidos": dias,
            "Cantidad_Prestamos": 0,
        }
    )

    print(f"\n¡Usuario {nombre} {apellido} registrado con éxito!")

## registrar un objeto

def registrar_item():
    """Permite registrar un nuevo artículo en el inventario"""
    limpiar_pantalla()
    print("--- REGISTRAR ARTÍCULO EN INVENTARIO ---")

    nombre = input("Nombre del articulo: ").strip()
    if len(nombre) < 3:
        print("El nombre del articulo debe tener al menos 3 caracteres")
        return

    print("\nCategorías disponibles:")
    print("1. Videojuegos\n2. Libros\n3. Música y video")
    print("4. Herramientas\n5. Dinero\n6. Misceláneo y varios")
    cat_opcion = input("Seleccione el número de categoría: ").strip()

    categorias = {
        "1": "Videojuegos",
        "2": "Libros",
        "3": "Música y video",
        "4": "Herramientas",
        "5": "Dinero",
        "6": "Misceláneo y varios",
    }

    if cat_opcion not in categorias:
        print("Categoría no válida")
        return
    categoria_seleccionada = categorias[cat_opcion]

    try:
        precio = float(input("Precio de compra: "))
        if precio < 0:
            print("El precio no puede ser negativo")
            return
    except ValueError:
        print("Debe ingresar un valor numérico")
        return

    print("\nEstado del articulo:")
    print("- Excelente (cerca a 4)\n- Bueno (carca a 3)")
    print("- Desgastado (cerca a 2)\n- Crítico (cerca a 1)")
    try:
        estado_val = float(
            input("Ingrese el estado del articulo (1.0 al 4.0)): ")
        )
        if not (1.0 <= estado_val <= 4.0):
            print("El valor debe estar entre 1.0 y 4.0.")
            return
    except ValueError:
        print("Debe ingresar un número entre 1.0 y 4.0")
        return

    # Clasificación lingüística según el valor difuso
    if estado_val >= 3.5:
        estado_texto = "Excelente"
    elif estado_val >= 2.5:
        estado_texto = "Bueno"
    elif estado_val >= 1.5:
        estado_texto = "Desgastado"
    else:
        estado_texto = "Crítico"

    # Generación de ID único combinado (Prefijo Categoría + Conteo)
    prefijo = categoria_seleccionada[:2].upper()
    id_item = f"{prefijo}-{len(items_db) + 101}"

    # Guardamos los datos utilizando una Serie de Pandas
    items_db[id_item] = pd.Series(
        {
            "Nombre": nombre,
            "Categoria": categoria_seleccionada,
            "Precio": precio,
            "Estado_Difuso": estado_texto,
            "Disponible": True,
        }
    )

    print(f"\n¡Artículo registrado con ID único: {id_item}!")

## registrar prestamo

def registrar_prestamo():
    """Genera un nuevo registro vinculando un articulo disponible con un
     usuario previamente registrado en la plataforma"""
    limpiar_pantalla()
    print("--- REGISTRAR PRÉSTAMO ---")

    if not usuarios_db:
        print("No hay usuarios registrados. Registre un usuario primero.")
        return

    if not items_db:
        print("No hay articulos en el inventario")
        return

    doc_usuario = input("Ingrese el documento del amigo solicitante: ").strip()
    if doc_usuario not in usuarios_db:
        print("El usuario no existe. Debe registrarlo primero en el sistema" )
        return

    ## Mostrar articulos disponibles
    print("\n--- Artículos Disponibles ---")
    items_disponibles = {
        k: v for k, v in items_db.items() if v["Disponible"] == True
    }

    if not items_disponibles:
        print("Lo sentimos, no hay artículos disponibles en este momento.")
        return

    df_items = pd.DataFrame(items_disponibles).T
    print(df_items[["Nombre", "Categoria", "Estado_Difuso"]])

    id_solicitado = input("\nIngrese el ID del articulo que desea prestar: ").strip()
    if id_solicitado not in items_db or not items_db[id_solicitado]["Disponible"]:
        print("ID no válido o artículo no se encuentra disponible")
        return

    ## asignamos fecha actual

    fecha_prestamo = datetime.now()

    # Actualizar estados e historial
    items_db[id_solicitado]["Disponible"] = False
    usuarios_db[doc_usuario]["Cantidad_Prestamos"] += 1

    nuevo_prestamo = {
        "ID_Item": id_solicitado,
        "Doc_Usuario": doc_usuario,
        "Fecha_Prestamo": fecha_prestamo,
        "Activo": True,
    }
    prestamos_db.append(nuevo_prestamo)

    print(
        f"\nPréstamo registrado exitosamente el: {fecha_prestamo.strftime('%Y-%m-%d')}"
    )

## registrar devolucion


def registrar_devolucion():
    """Gestiona el retorno de un objeto. Si está dentro de los 30 días """
    limpiar_pantalla()
    print("--- REGISTRAR EVOLUCIÓN ---")

    doc_usuario = input("Documento del amigo que devuelve: ").strip()

    # Filtrar préstamos activos de este usuario
    prestamos_usuario = [
        p
        for p in prestamos_db
        if p["Doc_Usuario"] == doc_usuario and p["Activo"] == True
    ]

    if not prestamos_usuario:
        print("Info: El usuario consultado no tiene préstamos activos en el sistema" )
        return

    print("\nPréstamos activos detectados:")
    for i, p in enumerate(prestamos_usuario):
        item_nom = items_db[p["ID_Item"]]["Nombre"]
        print(
            f"{i + 1}. Ítem: {item_nom} (ID: {p['ID_Item']}) - Prestado el: {p['Fecha_Prestamo'].strftime('%Y-%m-%d')}"
        )

    try:
        opcion = int(input("\nSeleccione el número del préstamo: ")) - 1
        if not (0 <= opcion < len(prestamos_usuario)):
            print("Opción inválida")
            return
    except ValueError:
        print("Entrada no válida")
        return

    prestamo_seleccionado = prestamos_usuario[opcion]
    id_item = prestamo_seleccionado["ID_Item"]

    ## Calcular los días transcurridos
    fecha_actual = datetime.now()
    dias_transcurridos = (
        fecha_actual - prestamo_seleccionado["Fecha_Prestamo"]
    ).days

    print(f"\nDías con el artículo en posesión: {dias_transcurridos} días.")

    ## Si pasan más de 30 días, se realizara la venta obligatoria
    if dias_transcurridos > 30:
        print("\n[INCUMPLIMIENTO] Se procederá a generar Factura de Venta")
        precio_base = items_db[id_item]["Precio"]
        impuesto_conchudez = precio_base * 0.23
        total_pagar = precio_base + impuesto_conchudez

        ## Registrar venta en base de datos de administración
        ventas_db.append(
            {
                "ID_Item": id_item,
                "Comprador": doc_usuario,
                "Total": total_pagar,
                "Impuesto": impuesto_conchudez,
            }
        )

        # Generar Factura en TXT Plano
        nombre_archivo = (
            f"Factura_{usuarios_db[doc_usuario]['Nombre']}_{id_item}.txt"
        )
        with open(nombre_archivo, "w", encoding="utf-8") as f:
            f.write("==================================================\n")
            f.write("           FACTURA DE VENTA - PRESTAMIGOS          \n")
            f.write("==================================================\n")
            f.write(f"Fecha de Emisión: {fecha_actual.strftime('%Y-%m-%d')}\n")
            f.write(
                f"Cliente : {usuarios_db[doc_usuario]['Nombre']} {usuarios_db[doc_usuario]['Apellido']}\n"
            )
            f.write(f"ID Artículo: {id_item}\n")
            f.write(f"Descripción: {items_db[id_item]['Nombre']}\n")
            f.write(f"Motivo: Exceder el tiempo límite de préstamo estipulado.\n")
            f.write("--------------------------------------------------\n")
            f.write(f"Subtotal: ${precio_base:,.2f}\n")
            f.write(f"Impuesto (23%):  ${impuesto_conchudez:,.2f}\n")
            f.write("--------------------------------------------------\n")
            f.write(f"TOTAL A PAGAR:                 ${total_pagar:,.2f}\n")
            f.write("==================================================\n")

        print(
            f"Factura de venta generada exitosamente como archivo: '{nombre_archivo}'."
        )

    else:
        ## Devolución Normal: Se genera Certificado de Devolución tradicional
        nombre_archivo = f"{usuarios_db[doc_usuario]['Nombre']}{fecha_actual.strftime('%Y%m%d')}{id_item}.txt"
        with open(nombre_archivo, "w", encoding="utf-8") as f:
            f.write("==================================================\n")
            f.write("       CERTIFICADO DE DEVOLUCIÓN - PRESTAMIGOS     \n")
            f.write("==================================================\n")
            f.write(
                f"Prestador: {usuarios_db[doc_usuario]['Nombre']} {usuarios_db[doc_usuario]['Apellido']}\n"
            )
            f.write(f"Artículo devuelto: {items_db[id_item]['Nombre']}\n")
            f.write(f"ID del Ítem: {id_item}\n")
            f.write(
                f"Fecha de Préstamo: {prestamo_seleccionado['Fecha_Prestamo'].strftime('%Y-%m-%d')}\n"
            )
            f.write(f"Fecha de Devolución: {fecha_actual.strftime('%Y-%m-%d')}\n")
            f.write(f"Estado de entrega verificado.\n")
            f.write("==================================================\n")

        print(f"Devolución procesada : '{nombre_archivo}'.")

    ## En ambos casos, el préstamo deja de estar activo
    prestamo_seleccionado["Activo"] = False
    items_db[id_item]["Disponible"] = True

## informacion

def consultar_items_mas_30_dias():
    """Filtra y muestra los artículos que llevan prestados más de 30 días
    calendario"""
    limpiar_pantalla()
    print("--- ARTÍCULOS CON MÁS DE 30 DÍAS DE PRÉSTAMO ---")

    registros = []
    fecha_actual = datetime.now()

    for p in prestamos_db:
        if p["Activo"]:
            dias = (fecha_actual - p["Fecha_Prestamo"]).days
            if dias > 30:
                registros.append(
                    {
                        "ID_Item": p["ID_Item"],
                        "Nombre_Item": items_db[p["ID_Item"]]["Nombre"],
                        "Doc_Amigo": p["Doc_Usuario"],
                        "Nombre_Amigo": usuarios_db[p["Doc_Usuario"]]["Nombre"],
                        "Dias_Transcurridos": dias,
                    }
                )

    if not registros:
        print("Excelente noticia: Ningún amigo supera los 30 días de retraso.")
    else:
        df = pd.DataFrame(registros)
        # Ordenamos de mayor a menor cantidad de días de retraso
        df = df.sort_values(by="Dias_Transcurridos", ascending=False)
        print(df.to_string(index=False))


def consultar_articulos_prestados():
    """Muestra una lista ordenada de todos los préstamos actuales"""
    limpiar_pantalla()
    print("--- ARTÍCULOS PRESTADOS ---")

    registros = []
    fecha_actual = datetime.now()

    for p in prestamos_db:
        if p["Activo"]:
            dias = (fecha_actual - p["Fecha_Prestamo"]).days
            registros.append(
                {
                    "ID": p["ID_Item"],
                    "Artículo": items_db[p["ID_Item"]]["Nombre"],
                    "Categoría": items_db[p["ID_Item"]]["Categoria"],
                    "Prestado A": usuarios_db[p["Doc_Usuario"]]["Nombre"],
                    "Días Transcurridos": dias,
                }
            )

    if not registros:
        print("No hay ningún artículo prestado actualmente.")
    else:
        df = pd.DataFrame(registros)
        df = df.sort_values(by="Días Transcurridos", ascending=False)
        print(df.to_string(index=False))

        ## exportacion de datos
        df.to_csv("Reporte_Prestamos_Activos.csv", index=False, encoding="utf-8")
        print("\n[Sistema] Datos actualizados y exportados a 'Reporte_Prestamos_Activos.csv'.")

## Administrador

def submenu_administrador():
    """Despliega la información de control interno del programa tras verificar
    las credenciales asignadas.
    """
    limpiar_pantalla()
    print("--- INGRESO AL MÓDULO ADMINISTRADOR ---")
    usuario = input("Usuario: ")
    contrasena = input("Contraseña: ")

    if usuario != ADMIN_USER or contrasena != ADMIN_PASS:
        print("Acceso denegado: Datos de administración incorrectas.")
        return

    while True:
        limpiar_pantalla()
        print("==================================================")
        print("        PRESTAMIGOS - PANEL DE ADMINISTRACIÓN     ")
        print("==================================================")
        print("1. Ver Total de Préstamos Registrados")
        print("2. Ver Total de Ítems Devueltos Correctamente")
        print("3. Ver Total de Ventas Realizadas e Impuestos")
        print("4. Ver Lista Completa de Usuarios ")
        print("5. Ver Usuario con Mayor y Menor cantidad de Préstamos")
        print("6. Volver al Menú Principal")
        print("==================================================")

        opcion = input("Seleccione una opción: ").strip()

        if opcion == "1":
            print(f"\nTotal históricos de préstamos: {len(prestamos_db)}")
        elif opcion == "2":
            devueltos = len([p for p in prestamos_db if not p["Activo"]])
            print(f"\nTotal histórico de ítems devueltos: {devueltos}")
        elif opcion == "3":
            if not ventas_db:
                print("\nNo se han registrado ventas por morosidad.")
            else:
                df_ventas = pd.DataFrame(ventas_db)
                print("\n--- Historial de Ventas Coactivas ---")
                print(df_ventas)
                print(f"\nRecaudo Total: ${df_ventas['Total'].sum():,.2f}")
                print(f"Total Impuestos (23%): ${df_ventas['Impuesto'].sum():,.2f}")
        elif opcion == "4":
            if not usuarios_db:
                print("\nNo hay usuarios en la base de datos.")
            else:
                df_usuarios = pd.DataFrame(usuarios_db).T
                print("\n--- Lista de Amigos Registrados ---")
                print(df_usuarios[["Nombre", "Apellido", "Correo"]])
        elif opcion == "5":
            if not usuarios_db:
                print("\nNo hay datos de usuarios disponibles.")
            else:
                df_u = pd.DataFrame(usuarios_db).T
                max_p = df_u.loc[df_u["Cantidad_Prestamos"].idxmax()]
                min_p = df_u.loc[df_u["Cantidad_Prestamos"].idxmin()]
                print(
                    f"\nMayor cantidad de préstamos: {max_p['Nombre']} ({max_p['Cantidad_Prestamos']})"
                     )
                print(
                    f"Menor cantidad de préstamos: {min_p['Nombre']} ({min_p['Cantidad_Prestamos']})"
                     )
        elif opcion == "6":
            break
        else:
            print("Opción no válida.")

## menu principal


def menu_principal():
    """Función de arranque que dibuja el menú de consola visualmente
    amigable e interactúa con el usuario final"""
    while True:
        limpiar_pantalla()
        print("==================================================")
        print("             BIENVENIDO A PRESTAMIGOS             ")
        print("==================================================")
        print("1. Registrar Usuario ")
        print("2. Registrar Objeto a Inventario")
        print("3. Registrar Préstamo")
        print("4. Registrar Devolución ")
        print("5. Consultar Articulos con más de 30 días")
        print("6. Consultar Artículos Prestados Actuales")
        print("7. Módulo Administrador")
        print("8. Salir del Programa")
        print("==================================================")

        opcion = input("Seleccione una opción (1-8): ").strip()

        if opcion == "1":
            registrar_usuario()
        elif opcion == "2":
            registrar_item()
        elif opcion == "3":
            registrar_prestamo()
        elif opcion == "4":
            registrar_devolucion()
        elif opcion == "5":
            consultar_items_mas_30_dias()
        elif opcion == "6":
            consultar_articulos_prestados()
        elif opcion == "7":
            submenu_administrador()
        elif opcion == "8":
            print("\n¡Gracias por utilizar Prestamigos! Hasta pronto.")
            break
        else:
            print("\nOpción inválida. Intente de nuevo.")


## inicio
if __name__ == "__main__":
    menu_principal()

             BIENVENIDO A PRESTAMIGOS             
1. Registrar Usuario 
2. Registrar Objeto a Inventario
3. Registrar Préstamo
4. Registrar Devolución 
5. Consultar Articulos con más de 30 días
6. Consultar Artículos Prestados Actuales
7. Módulo Administrador
8. Salir del Programa
